In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.decomposition import PCA
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# GPU diagnostics
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN") 


PyTorch: 2.10.0+cu128
CUDA available: True
Device: Tesla T4
VRAM: 15.6 GB


In [3]:
from pathlib import Path

print("Folders in /kaggle/input:")
for p in Path("/kaggle/input").iterdir():
    print(" ", p)

print("\nJSON files:")
for p in Path("/kaggle/input").rglob("*.jsonl"):
    print(" ", p)

Folders in /kaggle/input:
  /kaggle/input/datasets

JSON files:
  /kaggle/input/datasets/saksheee/emotion-stories1/neutral_texts.jsonl
  /kaggle/input/datasets/saksheee/emotion-stories1/emotion_stories.jsonl


In [4]:
# Fixed Parameters
EMOTIONS = ["desperate", "calm", "happy", "afraid", "angry"]
N_STORIES_PER_EMOTION = 60
N_NEUTRAL_TEXTS = 100
SWEEP_LAYERS = [8, 10, 13, 15, 17, 19, 21, 24, 26, 29]
D_MODEL = 4096
SKIP_FIRST_N_TOKENS = 50
PCA_VARIANCE_THRESHOLD = 0.50
POSITIVE_EMOTIONS = ["happy", "calm"]
NEGATIVE_EMOTIONS = ["desperate", "afraid", "angry"]

In [8]:
#time to load the data in
import json

DATA_DIR = Path("/kaggle/input/datasets/saksheee/emotion-stories1")
emotion_stories = {e: [] for e in EMOTIONS}
with open(DATA_DIR / "emotion_stories.jsonl", "r") as f:
    for line in f:
        obj = json.loads(line.strip())
        emotion_stories[obj["emotion"]].append(obj["text"])

for e in EMOTIONS:
    assert len(emotion_stories[e]) == N_STORIES_PER_EMOTION, \
        f"{e}: expected {N_STORIES_PER_EMOTION}, got {len(emotion_stories[e])}"
    print(f"{e}: {len(emotion_stories[e])} stories")


neutral_texts = []
with open(DATA_DIR / "neutral_texts.jsonl", "r") as f:
    for line in f:
        obj = json.loads(line.strip())
        neutral_texts.append(obj["text"])
        
assert len(neutral_texts) == N_NEUTRAL_TEXTS, \
    f"Neutral: expected {N_NEUTRAL_TEXTS}, got {len(neutral_texts)}"
print(f"Neutral texts: {len(neutral_texts)}")

assert len(neutral_texts) == N_NEUTRAL_TEXTS, \
    f"Neutral: expected {N_NEUTRAL_TEXTS}, got {len(neutral_texts)}"
print(f"Neutral texts: {len(neutral_texts)}")


for e in EMOTIONS:
    print(f"\n[{e}] {emotion_stories[e][0][:120]}...")

desperate: 60 stories
calm: 60 stories
happy: 60 stories
afraid: 60 stories
angry: 60 stories
Neutral texts: 100
Neutral texts: 100

[desperate] Marcus stared at the email timestamp—he had forty minutes before the server wiped his three years of unrecovered code. H...

[calm] The server room caught fire during Marcus's overnight shift, smoke pouring through the vents while alarms screamed. He p...

[happy] Marcus burst through the office door screaming his promotion before anyone could ask. He sprinted around cubicles high-f...

[afraid] Marcus sat in the fluorescent-lit break room, leg bouncing under the table. In ten minutes he would be called into the c...

[angry] Marcus sat in the quarterly review, jaw tight, as his manager took credit for the project he had spent three months buil...
